# Gemma 4 26B A4B Vision — FLN Worksheet Analyzer (Kaggle)

Sends the **full image** directly to Gemma 4 26B A4B (no OCR pipeline).  
Mixture-of-Experts architecture — only ~4B active params per token for fast inference.  
Includes **intelligent preprocessing** that auto-activates on blurry/small images.

---
**Setup**  
1. Settings → Accelerator → **GPU T4 x2**  
2. Add-ons → Secrets → Add **HF_TOKEN** (your Hugging Face token)  
3. Upload images via **Add Data** button (top right)  
4. Cell → Run All

---
**Note:** First run downloads the GGUF model (~17 GB, ~15 min on T4). Cached in `/root/gguf_cache/` (root partition has ~66 GB free).  
Model is split across both T4 GPUs (32 GB VRAM total) via llama.cpp's automatic multi-GPU support.

In [ ]:
import shutil, subprocess, os

shutil.rmtree("/kaggle/working/", ignore_errors=True)
try:
    shutil.rmtree("/tmp/", ignore_errors=True)
    os.makedirs("/tmp/", exist_ok=True)
except: pass
subprocess.run(["pip", "cache", "purge"], capture_output=True)

total, used, free = shutil.disk_usage("/kaggle/")
print(f"Disk free: {free // (1024**3)} GB")

In [ ]:
# Cell 1: GPU check + Install deps
import os, shutil, time, json, re, base64, zipfile
from pathlib import Path
from IPython.display import display, Image as IPyImage

import torch
if not torch.cuda.is_available():
    raise SystemExit('NO GPU. Go to Settings > Accelerator > GPU, then Run all again.')
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

OUTPUT_DIR = "/kaggle/working/FLN_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Outputs: {OUTPUT_DIR}")

!pip install opencv-python -q 2>&1 | tail -1
import cv2, numpy as np
print("Ready")

In [ ]:
# Cell 2: Hugging Face Login
from huggingface_hub import login

token = os.environ.get('HF_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass

if not token:
    raise ValueError('HF_TOKEN not found. Set it in Add-ons > Secrets.')

login(token)
print("Logged in")

In [ ]:
# Cell 3: Load Gemma 4 26B A4B (MoE)
import subprocess, sys

torch_ver = torch.version.cuda or ''
print(f"Torch CUDA: {torch_ver}")

wheel_map = {"12.1": "cu121", "12.2": "cu122", "12.3": "cu123",
             "12.4": "cu124", "12.5": "cu125", "12.6": "cu126"}
cuda_wheel = wheel_map.get(torch_ver, 'cu124')
print(f"Using {cuda_wheel} wheel")

print("Installing llama-cpp-python...")
sys.stdout.flush()
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'llama-cpp-python',
    f'--extra-index-url', f'https://abetlen.github.io/llama-cpp-python/whl/{cuda_wheel}',
    '--force-reinstall', '--no-cache-dir', '-q'
])
print("llama-cpp-python installed")

from llama_cpp import Llama
from llama_cpp.llama_chat_format import Gemma4ChatHandler

CACHE_DIR = "/root/gguf_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Extra cleanup before downloading 21 GB model
import shutil, subprocess
CHAT_HANDLER = Gemma4ChatHandler.from_pretrained(
    repo_id="unsloth/gemma-4-26B-A4B-it-GGUF",
    filename="mmproj-F16.gguf",
    local_dir=CACHE_DIR, verbose=False,
)
print("Chat handler OK")

start = time.time()
llm = Llama.from_pretrained(
    repo_id="unsloth/gemma-4-26B-A4B-it-GGUF",
    filename="gemma-4-26B-A4B-it-UD-Q5_K_XL.gguf",
    local_dir=CACHE_DIR,
    chat_handler=CHAT_HANDLER,
    n_gpu_layers=-1, n_ctx=8192,
    flash_attn=True, verbose=False,
)
t = (time.time()-start)/60
print(f"Model loaded in {t:.1f} min")

In [ ]:
# Cell 4: Image preprocessing

def preprocess(img: np.ndarray) -> tuple:
    h, w = img.shape[:2]
    orig_h, orig_w = h, w
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    result = img.copy()
    applied = []

    is_blurry = lap_var < 80
    is_small = min(h, w) < 800
    is_low_contrast = (float(gray.max()) - float(gray.min())) < 100

    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi/180, 200)
    angle = 0.0
    if lines is not None:
        angles = []
        for line in lines:
            theta = line[0][1]
            deg = np.degrees(theta) - 90
            if abs(deg) < 30:
                angles.append(deg)
        if angles:
            angle = np.median(angles)
    is_skewed = abs(angle) > 3.0

    if is_skewed:
        M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
        result = cv2.warpAffine(result, M, (w, h),
                               flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        applied.append(f"deskew {angle:.1f} deg")
        h, w = result.shape[:2]

    if is_blurry:
        blurred = cv2.GaussianBlur(result, (0, 0), 3.0)
        sharp = cv2.addWeighted(result, 1.5, blurred, -0.5, 0)
        sharp = np.clip(sharp, 0, 255).astype(np.uint8)
        new_var = cv2.Laplacian(cv2.cvtColor(sharp, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var()
        if new_var > lap_var:
            result = sharp
            applied.append("unsharp")

    if is_low_contrast or is_blurry:
        lab = cv2.cvtColor(result, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        result = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)
        applied.append("CLAHE")

    if is_small:
        scale = max(1.0, 800 / min(h, w))
        if scale > 1.1:
            result = cv2.resize(result, None, fx=scale, fy=scale,
                               interpolation=cv2.INTER_CUBIC)
            applied.append(f"upscale {scale:.1f}x")

    info = {"blurry": is_blurry, "skewed": is_skewed, "small": is_small,
            "low_contrast": is_low_contrast, "lap_var": round(lap_var, 1),
            "angle": round(angle, 1), "original_size": f"{orig_w}x{orig_h}",
            "applied": applied}
    return result, info

In [ ]:
# Cell 5: Enhanced FLN Prompt + analyze function

SYSTEM_PROMPT = """You are an expert FLN (Foundational Literacy and Numeracy) curriculum analyst.
You specialize in analyzing worksheets for children aged 3-8 (Pre-K to Grade 3)
following the NIPUN Bharat / NCF Foundational Stage framework.

Analyze the COMPLETE worksheet image systematically across these dimensions:

---
DIMENSION 1 — WORKSHEET IDENTITY
- What is the main instruction/heading?
- What type of question is this? (counting, addition, letter recognition, tracing, etc.)
- What subject domain? (numeracy, literacy, fine motor, mixed)
- What grade level is it designed for?

DIMENSION 2 — CURRICULUM ALIGNMENT
- What specific FLN concept is being taught?
- What is the sub-concept / specific skill?
- What learning outcome does this worksheet target?
- What Bloom's taxonomy level? (Remembering / Understanding / Applying / Analyzing)
- What specific skills are being developed? (be precise and comprehensive)
- What prior knowledge does the child need?

DIMENSION 3 — TASK ANALYSIS
- What exactly should the student do? (write, circle, match, color, trace, cut, draw)
- How many questions/items are on the worksheet?
- Is there an example provided?
- What is the answer format? (fill_in_blank, circle, match, draw, color, trace, cut_paste, write_number)
- Is the instruction clear for a young child?
- What number range or letter set is covered? (if applicable)

DIMENSION 4 — VISUAL & DESIGN ANALYSIS
- Describe all visual elements in detail (layout, images, illustrations, boxes, lines)
- What is the visual density? (simple / moderate / complex)
- What is the quality of illustrations? (clear and engaging / adequate / unclear)
- What is the page layout? (structured grid / free-form / mixed)
- Does the design support the learning goal?
- What language is the worksheet in?
- Does the content feel culturally relevant and relatable for Indian children?

DIMENSION 5 — PEDAGOGICAL QUALITY
- Does the worksheet provide scaffolding? (examples, gradual difficulty, visual aids)
- Does it use multiple representations? (pictures + numbers + words)
- What common misconceptions might arise? How does the worksheet address them?
- What is the difficulty level? (easy / medium / hard)
- What is the assessment purpose? (formative / diagnostic / practice / summative)
- How could a teacher adapt this for different learners? (struggling / advanced)

DIMENSION 6 — CONFIDENCE & REASONING
- How confident are you in your analysis? (0-100)
- Explain your reasoning with specific visual evidence from the worksheet.
- Note any ambiguities or uncertainty.
- Provide teacher guidance notes for effectively using this worksheet.

---
Important Rules:
* Use visual evidence as the primary source of truth.
* Read all text carefully — it contains critical instructions.
* Be precise and avoid guessing. If uncertain, note your uncertainty.
* Prefer the most specific educational concept visible.
* If students connect items with lines or draw arrows, classify as Matching.
* Do not classify as Patterns unless a repeating sequence is clearly visible.
* Do not classify as Mathematics unless numbers or operations are clearly present.
* For literacy worksheets: note script (Devanagari, Roman, etc.) and language.
* For numeracy worksheets: specify the exact number range.
* For mixed worksheets: identify the primary and secondary domains.
* Consider the Indian FLN context (NIPUN Bharat, NCF 2022).

Respond ONLY with valid JSON (no markdown, no code fences):
{
  "question_heading": "",
  "question_type": "Counting|Number Recognition|Missing Numbers|Ascending Order|Descending Order|Addition|Subtraction|Shapes|Patterns|Matching|Classification|Logical Reasoning|Tracing|Comparison|Skip Counting|Finger Counting|Review Assessment|Coloring|Writing Practice|Word Problem|Letter Recognition|Phonics|Vocabulary|Reading Comprehension|Sentence Writing|Measurement|Data Handling|Money|Time|Fractions",
  "learning_outcome": "",
  "student_action": "",
  "answer_format": "fill_in_blank|circle|match|draw|color|trace|cut_paste|write_number|underline|connect|paste|read_aloud",
  "visual_elements": "",
  "number_range": "",
  "has_example": false,
  "question_count": 0,
  "worksheet_type": "numeracy|literacy|fine_motor|mixed",
  "difficulty": "easy|medium|hard",
  "concept": "",
  "sub_concept": "",
  "estimated_grade": "Pre-K|Kindergarten|Grade 1|Grade 2|Grade 3",
  "bloom_level": "Remembering|Understanding|Applying|Analyzing",
  "skills": [],
  "language": "",
  "script": "",
  "instruction_clarity": "clear|moderate|unclear",
  "visual_density": "simple|moderate|complex",
  "scaffolding_details": "",
  "multiple_representations": false,
  "assessment_purpose": "formative|diagnostic|practice|summative",
  "prior_knowledge_required": "",
  "common_misconceptions": [],
  "cultural_context": "",
  "teacher_guidance": "",
  "adaptation_suggestions": "",
  "cross_curricular_links": "",
  "all_text_detected": "",
  "confidence_score": 0,
  "reasoning": ""
}

Confidence Guidelines:
90-100 = very clear visual evidence, unambiguous
70-89 = mostly clear, minor uncertainty
50-69 = partial evidence, multiple interpretations possible
below 50 = uncertain or poor image quality, significant guesswork
"""

def analyze_image(llm, img: np.ndarray, name: str) -> dict:
    _, buffer = cv2.imencode(".png", cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    b64 = base64.b64encode(buffer).decode("utf-8")
    resp = llm.create_chat_completion(
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            {"type": "text", "text": SYSTEM_PROMPT + "\n\nAnalyze this FLN worksheet image."},
        ]}],
        max_tokens=8192, temperature=0.1,
    )
    raw = resp["choices"][0]["message"]["content"]
    parsed = None; error = None
    cleaned = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    cleaned = re.sub(r'```json\s*|```\s*', '', cleaned).strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        a, b = cleaned.find('{'), cleaned.rfind('}')
        if a != -1 and b > a:
            try:
                parsed = json.loads(cleaned[a:b+1])
            except json.JSONDecodeError:
                error = str(e)
        else:
            error = str(e)
    # Ensure backward compatibility — fill any missing new fields with defaults
    if parsed and isinstance(parsed, dict):
        defaults = {
            "question_count": parsed.get("question_count", 0),
            "language": parsed.get("language", ""),
            "script": parsed.get("script", ""),
            "instruction_clarity": parsed.get("instruction_clarity", ""),
            "visual_density": parsed.get("visual_density", ""),
            "scaffolding_details": parsed.get("scaffolding_details", ""),
            "multiple_representations": parsed.get("multiple_representations", False),
            "assessment_purpose": parsed.get("assessment_purpose", ""),
            "prior_knowledge_required": parsed.get("prior_knowledge_required", ""),
            "common_misconceptions": parsed.get("common_misconceptions", []),
            "cultural_context": parsed.get("cultural_context", ""),
            "teacher_guidance": parsed.get("teacher_guidance", ""),
            "adaptation_suggestions": parsed.get("adaptation_suggestions", ""),
            "cross_curricular_links": parsed.get("cross_curricular_links", ""),
        }
        parsed.update(defaults)
    return {"file": name, "raw": raw, "parsed": parsed, "error": error}


In [ ]:
# Cell 6: Find images + extract zips + Run analysis

ZEXT = "/kaggle/working/zip_extract"
if os.path.exists(ZEXT):
    shutil.rmtree(ZEXT)
os.makedirs(ZEXT, exist_ok=True)

image_paths = []
for root, _, files in os.walk("/kaggle/input"):
    for fn in sorted(files):
        fp = os.path.join(root, fn)
        if fn.lower().endswith(".zip"):
            with zipfile.ZipFile(fp) as z:
                z.extractall(ZEXT)
            print(f"  Extracted: {fn}")
        elif fn.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff")):
            image_paths.append(fp)

for root, _, files in os.walk(ZEXT):
    for fn in sorted(files):
        if fn.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff")):
            image_paths.append(os.path.join(root, fn))

if not image_paths:
    print("No images found in /kaggle/input.")
    print("Upload images using the Add Data button (top right), then re-run this cell.")
else:
    print(f"Found {len(image_paths)} image(s)")

all_results = []
for idx, img_path in enumerate(image_paths, 1):
    name = Path(img_path).name
    stem = Path(name).stem
    img_dir = os.path.join(OUTPUT_DIR, stem)
    os.makedirs(img_dir, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"[{idx}/{len(image_paths)}] {name}")
    print(f"{'='*60}")

    img = cv2.imread(img_path)
    if img is None:
        print(f"  FAILED: cannot read {name}")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print(f"  Input: {img.shape[1]}x{img.shape[0]}")

    img, info = preprocess(img)
    if info["applied"]:
        print(f"  Preprocessing: {', '.join(info['applied'])}")
    else:
        print("  Preprocessing: none needed")

    enhanced_path = os.path.join(img_dir, name)
    cv2.imwrite(enhanced_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    display(IPyImage(img_path))

    result = analyze_image(llm, img, name)
    all_results.append(result)

    print(f"\n{'='*60}")
    print("MODEL OUTPUT:")
    print(f"{'='*60}")
    print(result["raw"])
    print(f"{'='*60}")

    if result["error"]:
        print(f"\nJSON PARSE FAILED: {result['error']}")
    elif result["parsed"]:
        print("\nPARSED JSON:")
        print(json.dumps(result["parsed"], indent=2))
        out_name = Path(name).stem + "_result.json"
        out_path = os.path.join(img_dir, out_name)
        with open(out_path, "w") as f:
            json.dump(result["parsed"], f, indent=2)
        print(f"\nSaved: {out_path}")

if image_paths:
    ok = sum(1 for r in all_results if r["parsed"] is not None)
    fail = sum(1 for r in all_results if r["error"] or r["parsed"] is None)
    print(f"\n{'='*60}")
    print(f"DONE: {len(all_results)} total, {ok} ok, {fail} failed")
    print(f"{'='*60}")
    manifest = {"batch": [{"file": r["file"],
        "confidence": r["parsed"].get("confidence_score") if r["parsed"] and isinstance(r["parsed"], dict) else None,
        "question_type": r["parsed"].get("question_type") if r["parsed"] and isinstance(r["parsed"], dict) else None}
        for r in all_results]}
    manifest_path = os.path.join(OUTPUT_DIR, "batch_manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"Manifest: {manifest_path}")

In [ ]:
!zip -r /kaggle/working/FLN_Results.zip /kaggle/working/FLN_Results